In [36]:
--The Foodie-Fi team wants you to create a new payments table for the year 2020 that includes amounts paid by each customer in the subscriptions table with the following requirements:

---- monthly payments always occur on the same day of month as the original start_date of any monthly paid plan
---- upgrades from basic to monthly or pro plans are reduced by the current paid amount in that month and start immediately
---- upgrades from pro monthly to pro annual are paid at the end of the current billing period and also starts at the end of the month period
---- once a customer churns they will no longer make payments

-- Check all tables

-- plans
SELECT * FROM foodie_fi.plans

-- subsriptions
SELECT TOP 5 * FROM foodie_fi.subscriptions
SELECT COUNT(DISTINCT customer_id) as unique_customer FROM foodie_fi.subscriptions
SELECT COUNT(customer_id) as user_transaction FROM foodie_fi.subscriptions

-- foofie merge
DROP TABLE IF EXISTS #temp_foodie_merge;
SELECT s.customer_id,
    s.plan_id,
    p.plan_name,
    s.start_date,
    p.price
INTO #temp_foodie_merge
FROM foodie_fi.plans p 
INNER JOIN foodie_fi.subscriptions s 
ON p.plan_id = s.plan_id

SELECT TOP 5 * FROM #temp_foodie_merge


(5 rows affected)

(5 rows affected)

(1 row affected)

(1 row affected)

(2650 rows affected)

...

plan_id,plan_name,price
0,trial,0.00
1,basic monthly,9.90
2,pro monthly,19.90
3,pro annual,199.00
4,churn,NULL


customer_id,plan_id,start_date
1,0,2020-08-01
1,1,2020-08-08
2,0,2020-09-20
2,3,2020-09-27
3,0,2020-01-13


unique_customer
1000


user_transaction
2650


customer_id,plan_id,plan_name,start_date,price
1,0,trial,2020-08-01,0.00
1,1,basic monthly,2020-08-08,9.90
2,0,trial,2020-09-20,0.00
2,3,pro annual,2020-09-27,199.00
3,0,trial,2020-01-13,0.00


In [37]:
SELECT column_name, 
    data_type,
    character_maximum_length 
FROM information_schema.columns
WHERE table_name in ('plans','subscriptions')

(6 rows affected)

Total execution time: 00:00:00.067

column_name,data_type,character_maximum_length
plan_id,int,NULL
plan_name,varchar,13
price,decimal,NULL
customer_id,int,NULL
plan_id,int,NULL
start_date,date,NULL


In [38]:
SELECT *
FROM #temp_foodie_merge


(2650 rows affected)

Total execution time: 00:00:00.027

customer_id,plan_id,plan_name,start_date,price
1,0,trial,2020-08-01,0.00
1,1,basic monthly,2020-08-08,9.90
2,0,trial,2020-09-20,0.00
2,3,pro annual,2020-09-27,199.00
3,0,trial,2020-01-13,0.00
3,1,basic monthly,2020-01-20,9.90
4,0,trial,2020-01-17,0.00
4,1,basic monthly,2020-01-24,9.90
4,4,churn,2020-04-21,NULL
5,0,trial,2020-08-03,0.00


In [39]:
SELECT s.plan_journey,
    COUNT(s.customer_id) as customer_count
FROM
    (SELECT customer_id, 
        STRING_AGG(CAST(plan_id AS VARCHAR),',') as plan_journey
    FROM #temp_foodie_merge
    GROUP BY customer_id
    ) s 
GROUP BY s.plan_journey 


(14 rows affected)

Total execution time: 00:00:00.021

plan_journey,customer_count
"0,1",125
"0,1,2",138
"0,1,2,3",35
"0,1,2,4",41
"0,1,3",107
"0,1,3,4",3
"0,1,4",97
"0,2",178
"0,2,3",75
"0,2,3,4",1


In [40]:
DROP TABLE IF EXISTS #temp_merge_journey;
SELECT m.*,
    j.plan_journey
INTO #temp_merge_journey
FROM #temp_foodie_merge m
JOIN   
    (SELECT customer_id, 
            STRING_AGG(CAST(plan_id AS VARCHAR),',') as plan_journey
    FROM #temp_foodie_merge
    GROUP BY customer_id
    ) j 
ON j.customer_id = m.customer_id

SELECT TOP 5 * FROM #temp_merge_journey


(2650 rows affected)

(5 rows affected)

Total execution time: 00:00:00.252

customer_id,plan_id,plan_name,start_date,price,plan_journey
1,0,trial,2020-08-01,0.00,"0,1"
1,1,basic monthly,2020-08-08,9.90,"0,1"
3,0,trial,2020-01-13,0.00,"0,1"
3,1,basic monthly,2020-01-20,9.90,"0,1"
10,0,trial,2020-09-19,0.00,"0,2"


In [146]:
WITH cte_check_plan AS 
(
    SELECT l.*,
    DATEDIFF(DAY, l.lag_date, l.start_date) as diff_date
    FROM 
        (SELECT j.*,
            LAG(j.start_date, 1) OVER(PARTITION BY j.customer_id ORDER BY j.plan_id ASC, j.start_date ASC) AS lag_date
        FROM #temp_merge_journey j
        ) l
    WHERE DATEDIFF(DAY, l.lag_date, l.start_date) < 7
)
SELECT *
FROM cte_check_plan l
WHERE l.plan_journey = '0,1,2,3'

(1 row affected)

Total execution time: 00:00:00.072

customer_id,plan_id,plan_name,start_date,price,plan_journey,lag_date,diff_date
806,2,pro monthly,2020-05-13,19.90,"0,1,2,3",2020-05-09,4


In [42]:
WITH cte_check_plan AS 
(
    SELECT l.*,
    DATEDIFF(DAY, l.lag_date, l.start_date) as diff_date
    FROM 
        (SELECT j.*,
            LAG(j.start_date, 1) OVER(PARTITION BY j.customer_id ORDER BY j.plan_id ASC, j.start_date ASC) AS lag_date
        FROM #temp_merge_journey j
        ) l
    WHERE DATEDIFF(DAY, l.lag_date, l.start_date) = 7
)
SELECT l.plan_journey,
    COUNT(l.plan_journey) as plan_counts
FROM cte_check_plan l
GROUP BY l.plan_journey

(14 rows affected)

Total execution time: 00:00:00.026

plan_journey,plan_counts
"0,1",125
"0,1,2",139
"0,1,2,3",35
"0,1,2,4",41
"0,1,3",107
"0,1,3,4",3
"0,1,4",98
"0,2",178
"0,2,3",75
"0,2,3,4",1


## **Case 1: User upgrade plans to basic monthly/ pro-monthly (then churn or keep using)**

In [166]:
DROP TABLE IF EXISTS #temp_upgrade_monthly_annualy
SELECT j.*,
    LEAD(j.start_date, 1) OVER(PARTITION BY j.customer_id ORDER BY  j.start_date ASC) AS lead_date,
    LEAD(j.plan_id, 1) OVER(PARTITION BY j.customer_id ORDER BY j.plan_id ASC) AS lead_plan,
    LAG(j.price, 1) OVER(PARTITION BY j.customer_id ORDER BY j.price ASC) AS lag_price
INTO #temp_upgrade_monthly_annualy
FROM #temp_merge_journey j
WHERE  (j.plan_journey LIKE '%4' 
    OR j.plan_journey LIKE '%1'
    OR j.plan_journey LIKE '%2'
    OR j.plan_journey LIKE '%1,3'
    OR j.plan_journey = '0,3')
    AND j.plan_journey != '0,4' 


-- Testing
SELECT *
FROM #temp_upgrade_monthly_annualy
WHERE customer_id = 7

(2101 rows affected)

(3 rows affected)

Total execution time: 00:00:00.230

customer_id,plan_id,plan_name,start_date,price,plan_journey,lead_date,lead_plan,lag_price
7,0,trial,2020-02-05,0.00,"0,1,2",2020-02-12,1,NULL
7,1,basic monthly,2020-02-12,9.90,"0,1,2",2020-05-22,2,0.00
7,2,pro monthly,2020-05-22,19.90,"0,1,2",NULL,NULL,9.90


In [167]:
DROP TABLE IF EXISTS #temp_final_upgrade_monthly_annualy;
WITH lastest_day AS (
    SELECT customer_id,
        plan_id,
        start_date,
        (CASE 
            WHEN lead_plan = 4 THEN MAX(lead_date) OVER(PARTITION BY customer_id) 
            WHEN lead_plan IS NULL THEN CAST('2022-12-31' AS DATE)
            ELSE lead_date
        END) as end_date
    FROM #temp_upgrade_monthly_annualy
    WHERE plan_id != 4 AND plan_id != 0 
),
date_range AS (
    SELECT customer_id,
        plan_id,
        start_date,
        end_date
    FROM lastest_day

    UNION ALL

    SELECT 
        customer_id,
        plan_id, 
        DATEADD(MONTH, 1, start_date),
        end_date
    FROM date_range 
    WHERE DATEADD(MONTH, 1, start_date) <= end_date

),
subscription_monthly_annual AS (
    SELECT c.customer_id,
        c.plan_id,
        c.lead_plan,
        c.plan_name, 
        c.lag_price,
        (CASE 
            WHEN r.start_date IS NULL THEN c.start_date ELSE r.start_date 
        END) AS start_date,
        c.price,
        ROW_NUMBER() OVER(PARTITION BY c.customer_id, c.plan_id ORDER BY r.start_date ASC) as rn
    FROM date_range r
    RIGHT JOIN #temp_upgrade_monthly_annualy c
    ON c.customer_id = r.customer_id AND c.plan_id = r.plan_id
)
SELECT customer_id,
    plan_id,
    plan_name,
    start_date,
    (CASE
        WHEN plan_id = 2 AND rn = 1 THEN price - lag_price 
        WHEN plan_id = 3 AND rn = 1 THEN price - lag_price 
        WHEN plan_id = 3 AND rn != 1 THEN 0
        ELSE price
    END) AS price
INTO #temp_final_upgrade_monthly_annualy
FROM subscription_monthly_annual

--
SELECT * FROM #temp_final_upgrade_monthly_annualy
ORDER BY customer_id ASC, start_date ASC

Warning: Null value is eliminated by an aggregate or other SET operation.

(19648 rows affected)

(19648 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:00.531

customer_id,plan_id,plan_name,start_date,price
1,0,trial,2020-08-01,0.00
1,1,basic monthly,2020-08-08,9.90
1,1,basic monthly,2020-09-08,9.90
1,1,basic monthly,2020-10-08,9.90
1,1,basic monthly,2020-11-08,9.90
1,1,basic monthly,2020-12-08,9.90
1,1,basic monthly,2021-01-08,9.90
1,1,basic monthly,2021-02-08,9.90
1,1,basic monthly,2021-03-08,9.90
1,1,basic monthly,2021-04-08,9.90


**Case 2: User upgrade plans from pro-monthly to annual**

In [169]:
DROP TABLE IF EXISTS #temp_monthly_to_annual
SELECT j.*,
    LEAD(j.start_date, 1) OVER(PARTITION BY j.customer_id ORDER BY  j.start_date ASC) AS lead_date,
    LEAD(j.plan_id, 1) OVER(PARTITION BY j.customer_id ORDER BY j.plan_id ASC) AS lead_plan,
    LAG(j.price, 1) OVER(PARTITION BY j.customer_id ORDER BY j.price ASC) AS lag_price
INTO #temp_monthly_to_annual
FROM #temp_merge_journey j
WHERE j.plan_journey LIKE '%3%'
    AND NOT EXISTS (
        SELECT 1
        FROM #temp_upgrade_monthly_annualy u 
        WHERE j.customer_id = u.customer_id
    )
    


-- Testing
SELECT *
FROM #temp_monthly_to_annual


(365 rows affected)

(365 rows affected)

Total execution time: 00:00:00.134

customer_id,plan_id,plan_name,start_date,price,plan_journey,lead_date,lead_plan,lag_price
19,0,trial,2020-06-22,0.00,"0,2,3",2020-06-29,2,NULL
19,2,pro monthly,2020-06-29,19.90,"0,2,3",2020-08-29,3,0.00
19,3,pro annual,2020-08-29,199.00,"0,2,3",NULL,NULL,19.90
24,0,trial,2020-11-10,0.00,"0,2,3",2020-11-17,2,NULL
24,2,pro monthly,2020-11-17,19.90,"0,2,3",2021-04-17,3,0.00
24,3,pro annual,2021-04-17,199.00,"0,2,3",NULL,NULL,19.90
31,0,trial,2020-06-22,0.00,"0,2,3",2020-06-29,2,NULL
31,2,pro monthly,2020-06-29,19.90,"0,2,3",2020-11-29,3,0.00
31,3,pro annual,2020-11-29,199.00,"0,2,3",NULL,NULL,19.90
38,0,trial,2020-10-02,0.00,"0,2,3",2020-10-09,2,NULL


In [255]:
DROP TABLE IF EXISTS #temp_final_promonthly_proannual;
WITH lastest_day AS (
    SELECT customer_id,
        plan_id,
        price,
        start_date,
        (CASE 
            WHEN lead_plan = 4 THEN MAX(lead_date) OVER(PARTITION BY customer_id) 
            WHEN lead_plan IS NULL THEN CAST('2022-12-31' AS DATE)
            ELSE lead_date
        END) as end_date,
        lead_plan,
        lag_price,
        lead_date, 
        plan_name
    FROM #temp_monthly_to_annual
    WHERE plan_id != 4 AND plan_id != 0 
),
month_date_range AS (
    SELECT customer_id,
        plan_id,
        start_date,
        end_date
    FROM lastest_day

    UNION ALL

    SELECT 
        customer_id,
        plan_id, 
        DATEADD(DAY, 30, start_date),
        end_date
    FROM month_date_range
    WHERE DATEADD(DAY, 30, start_date) <= end_date

),
annual_date_range AS (
    SELECT customer_id,
        plan_id,
        price,
        start_date,
        end_date
    FROM lastest_day
    WHERE plan_id = 3

    UNION ALL

    SELECT 
        customer_id,
        plan_id, 
        price,
        DATEADD(DAY, 30 * 12, start_date),
        end_date
    FROM annual_date_range
    WHERE DATEADD(DAY, 30 * 12, start_date) < end_date
),
pro_month AS (
    SELECT c.customer_id,
        c.plan_id,
        c.lead_plan,
        c.plan_name, 
        c.lag_price,
        (CASE 
            WHEN r.start_date IS NULL THEN c.start_date ELSE r.start_date 
        END) AS start_date,
        c.price,
        ROW_NUMBER() OVER(PARTITION BY c.customer_id, c.plan_id ORDER BY r.start_date DESC) as rnd,
        ROW_NUMBER() OVER(PARTITION BY c.customer_id, c.plan_id ORDER BY r.start_date ASC) as rna
    FROM #temp_monthly_to_annual c 
    LEFT JOIN month_date_range r ON c.customer_id = r.customer_id AND c.plan_id = r.plan_id
),
pro_annual AS (
    SELECT c.customer_id,
        c.plan_id,
        c.lead_plan,
        c.plan_name, 
        c.lag_price,
        (CASE 
            WHEN r.start_date IS NULL THEN c.start_date ELSE r.start_date 
        END) AS start_date,
        c.price,
        ROW_NUMBER() OVER(PARTITION BY c.customer_id, c.plan_id ORDER BY r.start_date ASC) as rn
    FROM  #temp_monthly_to_annual c 
    LEFT JOIN annual_date_range r ON c.customer_id = r.customer_id AND c.plan_id = r.plan_id
)
SELECT DISTINCT p.customer_id,
    p.plan_id,
    p.plan_name,
    p.start_date,
    (CASE
        
        WHEN p.plan_id = 2 AND p.rna = 1 THEN p.price - p.lag_price 
        WHEN p.plan_id = 3 AND m.start_date = p.start_date THEN m.price
        WHEN p.plan_id = 3 AND p.rna != 1 THEN 0
        ELSE p.price
    END) AS price
INTO #temp_final_promonthly_proannual
FROM pro_annual m 
INNER JOIN pro_month p
ON m.customer_id = p.customer_id
    AND m.plan_id = p.plan_id
WHERE p.rnd != 1

Warning: Null value is eliminated by an aggregate or other SET operation.

(3456 rows affected)

Total execution time: 00:00:00.162

In [262]:
SELECT name, 
    create_date
FROM tempdb.sys.tables 

(8 rows affected)

Total execution time: 00:00:00.113

name,create_date
#temp_upgrade_monthly_______________________________________________________________________________________________000000000190,2026-04-05 17:15:33.647
#temp_010203________________________________________________________________________________________________________0000000000C7,2026-04-05 14:31:31.333
#temp_upgrade_monthly_annualy_______________________________________________________________________________________0000000001F1,2026-04-05 17:56:53.087
#temp_final_upgrade_monthly_annualy_________________________________________________________________________________0000000001F2,2026-04-05 17:57:02.417
#temp_monthly_to_annual_____________________________________________________________________________________________0000000001F8,2026-04-05 17:57:27.330
#temp_foodie_merge__________________________________________________________________________________________________0000000000B6,2026-04-05 14:31:22.620
#temp_final_promonthly_proannual____________________________________________________________________________________00000000021F,2026-04-05 19:43:08.813
#temp_merge_journey_________________________________________________________________________________________________0000000000BE,2026-04-05 14:31:27.457


In [264]:
WITH cte_merge AS (
    SELECT *
    FROM #temp_final_promonthly_proannual

    UNION ALL 

    SELECT *
    FROM #temp_final_upgrade_monthly_annualy

    UNION ALL

    SELECT customer_id,
        plan_id,
        plan_name,
        start_date,
        price
    FROM #temp_merge_journey
    WHERE plan_journey = '0,4'
)
SELECT * FROM cte_merge
ORDER BY customer_id, start_date ASC

(23288 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:00.126

customer_id,plan_id,plan_name,start_date,price
1,0,trial,2020-08-01,0.00
1,1,basic monthly,2020-08-08,9.90
1,1,basic monthly,2020-09-08,9.90
1,1,basic monthly,2020-10-08,9.90
1,1,basic monthly,2020-11-08,9.90
1,1,basic monthly,2020-12-08,9.90
1,1,basic monthly,2021-01-08,9.90
1,1,basic monthly,2021-02-08,9.90
1,1,basic monthly,2021-03-08,9.90
1,1,basic monthly,2021-04-08,9.90


In [261]:
DROP TABLE IF EXISTS #temp_churn_after_subscription

Commands completed successfully.

Total execution time: 00:00:00.014